# 1_veri — Türkiye Tüketim ve EAK Verisinin Hazırlanması

Bu notebook, Bitirme Projesi (Bonus) Faz 1'in tamamını sırayla çalıştırır:

- **A.** Kurulum ve veri indirme (önbellekli)
- **B.** Kalite kontrolü
- **C.** Birleştirme ve marj taraması
- **D.** EAK kırılma analizi
- **E.** Temizlik, işaretler ve Agent eşikleri

**Çıktılar:** `data/tr_tuketim_saatlik.csv`, `data/tr_eak_saatlik.csv`, `data/tr_birlesik_saatlik.csv`, `metrics/agent_esikleri.json`

**Ön koşul:** `data/ankara_tavg_gunluk.csv` dosyası `0_fizibilite.ipynb` ile üretilmiş olmalı.

## A. Kurulum ve Veri İndirme

### A1. Kurulum ve dayanıklı indirme fonksiyonları

- `cagir_sabirli`: 429 (çok fazla istek) hatasında 30, 60, 120, 240 saniye bekleyerek tekrar dener (üstel geri çekilme). Yeni oturumu (TGT) yalnızca 429 dışındaki hatalarda açar; TGT 2 saat geçerli.
- `aylik_indir`: Veriyi ay ay çeker, aylar arasında 1,5 saniye bekler. Tamamen boş sütunları (örneğin 2016-2022'deki `gunes`) birleştirmeden önce çıkarır; bu sütunlar birleşik tabloda yine NaN olarak görünür.
- `onbellekli`: Dosya varsa diskten okur, yoksa indirir. **Veriyi güncellemek için ilgili CSV'yi silip bu hücreleri tekrar çalıştırın.**

In [1]:
import os
import time
import json
import pandas as pd
from dotenv import load_dotenv
from eptr2 import EPTR2

KOK = r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus"
load_dotenv(os.path.join(KOK, "notepads", ".env"))


def yeni_istemci():
    return EPTR2(username=os.getenv("EPTR_USERNAME"), password=os.getenv("EPTR_PASSWORD"), use_dotenv=False)


istemci = {"eptr": yeni_istemci()}


def cagir_sabirli(anahtar, bas, bit, **ek):
    for deneme, bekle in enumerate([0, 30, 60, 120, 240], start=1):
        time.sleep(bekle)
        try:
            return istemci["eptr"].call(anahtar, start_date=bas, end_date=bit, **ek)
        except Exception as e:
            mesaj = str(e)
            print(f"{bas} deneme {deneme} hata: {mesaj[:80]}")
            if "429" not in mesaj:
                istemci["eptr"] = yeni_istemci()   # yalnizca yetki gibi hatalarda yeni oturum
    raise RuntimeError(f"{bas} indirilemedi")


def aylik_indir(anahtar, bas, bit, **ek):
    parcalar = []
    bit = pd.Timestamp(bit)
    for ay in pd.date_range(bas, bit, freq="MS"):
        son = min(ay + pd.offsets.MonthEnd(0), bit)
        parcalar.append(cagir_sabirli(anahtar, ay.strftime("%Y-%m-%d"), son.strftime("%Y-%m-%d"), **ek))
        time.sleep(1.5)
    return pd.concat([p.dropna(axis=1, how="all") for p in parcalar], ignore_index=True)


BASLANGIC = "2016-04-01"   # tuketim ve EAK verisinin ortak baslangici (0_fizibilite, Hucre 16 ve 19)
BITIS = (pd.Timestamp.today().normalize() - pd.Timedelta(days=1)).strftime("%Y-%m-%d")  # dun


def onbellekli(anahtar, dosya, **ek):
    yol = os.path.join(KOK, "data", dosya)
    if os.path.exists(yol):
        df = pd.read_csv(yol)
        print(f"{dosya}: onbellekten okundu ({len(df)} satir)")
    else:
        df = aylik_indir(anahtar, BASLANGIC, BITIS, **ek)
        df.to_csv(yol, index=False)
        print(f"{dosya}: indirildi ({len(df)} satir)")
    return df


print("Donem:", BASLANGIC, "->", BITIS)

Donem: 2016-04-01 -> 2026-09-28


### A2. Tüketim ve EAK

Dosyalar yoksa indirme birkaç dakika sürer (yaklaşık 250 istek). `cagir_sabirli` bir ayı beş denemede de indiremezse hata verir; böylece Şubat 2025'te yaşanan sessiz eksik ay durumu tekrarlanmaz.

In [2]:
tuketim = onbellekli("rt-cons", "tr_tuketim_saatlik.csv")
eak_tum = onbellekli("eak", "tr_eak_saatlik.csv", region="TR1")

tr_tuketim_saatlik.csv: onbellekten okundu (91968 satir)
tr_eak_saatlik.csv: onbellekten okundu (91968 satir)


## B. Kalite Kontrolü

### B1. Tekrar, eksik saat, boş değer ve tek saatlik sıçrama

**Sıçrama kontrolünün mantığı:** Bir saatte büyük bir artış ve hemen ardından büyük bir düşüş (ya da tersi) arar. PJM'deki 2008 anomalisi tam bu imzayı taşıyordu. Eşik olarak saatlik değişimlerin 4 standart sapması kullanılıyor; bu bir tarama sezgiseli, bulunan adaylar D bölümünde tek tek inceleniyor.

Fonksiyon ayrıca tarihleri datetime'a çevirir ve veriyi tarihe göre sıralar; sonraki bütün bölümler bu sıralamaya dayanır.

In [3]:
def kalite(df, sutun, ad):
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"])
    d = d.sort_values("date").reset_index(drop=True)
    tam = pd.date_range(d["date"].min(), d["date"].max(), freq="h")
    eksik = tam.difference(pd.DatetimeIndex(d["date"]))
    print(f"\n=== {ad} ===")
    print("Aralik:", d["date"].min(), "->", d["date"].max(), "| satir:", len(d))
    print("Tekrar eden saat:", d["date"].duplicated().sum())
    print("Eksik saat:", len(eksik), "| ilk 5:", list(eksik[:5]))
    print("Bos deger:", d[sutun].isna().sum())
    print(d[sutun].describe().round(0).to_string())
    fark = d[sutun].diff()
    esik = 4 * fark.abs().std()
    sicrama = d[((fark > esik) & (fark.shift(-1) < -esik)) | ((fark < -esik) & (fark.shift(-1) > esik))]
    print(f"Tek saatlik sicrama adayi (esik {esik:.0f} MWh):", len(sicrama))
    if len(sicrama):
        print(sicrama[["date", sutun]].head(10).to_string(index=False))
    return d


tuketim = kalite(tuketim, "consumption", "Tuketim")
eak_tum = kalite(eak_tum, "toplam", "EAK")


=== Tuketim ===
Aralik: 2016-04-01 00:00:00+03:00 -> 2026-09-27 23:00:00+03:00 | satir: 91968
Tekrar eden saat: 0
Eksik saat: 0 | ilk 5: []
Bos deger: 0
count    91968.0
mean     35996.0
std       6458.0
min      15333.0
25%      31290.0
50%      35581.0
75%      40273.0
max      59504.0
Tek saatlik sicrama adayi (esik 3964 MWh): 0

=== EAK ===
Aralik: 2016-04-01 00:00:00+03:00 -> 2026-09-27 23:00:00+03:00 | satir: 91968
Tekrar eden saat: 0
Eksik saat: 0 | ilk 5: []
Bos deger: 0
count    91968.0
mean     60500.0
std       6916.0
min      42117.0
25%      56052.0
50%      60145.0
75%      64963.0
max      89038.0
Tek saatlik sicrama adayi (esik 3182 MWh): 46
                     date   toplam
2025-04-19 23:00:00+03:00 63357.71
2025-04-20 00:00:00+03:00 57866.26
2025-05-02 15:00:00+03:00 69593.93
2025-05-09 07:00:00+03:00 66409.29
2025-05-09 14:00:00+03:00 67466.50
2025-05-13 19:00:00+03:00 67460.91
2025-05-16 07:00:00+03:00 67889.49
2025-05-22 15:00:00+03:00 73780.03
2025-05-24 10:00:0

### B2. Tüketimin uç değerleri

Komşu saatler düzgün bir iniş veya çıkış gösteriyorsa değer gerçektir. Tek bir saat komşularından çok farklıysa kayıt hatasıdır.

In [4]:
for etiket, idx in [("En dusuk", tuketim["consumption"].idxmin()), ("En yuksek", tuketim["consumption"].idxmax())]:
    print(f"\n{etiket}: {tuketim.loc[idx, 'date']} -> {tuketim.loc[idx, 'consumption']:.0f} MWh")
    print(tuketim.loc[idx - 3:idx + 3, ["date", "consumption"]].to_string(index=False))


En dusuk: 2020-05-25 07:00:00+03:00 -> 15333 MWh
                     date  consumption
2020-05-25 04:00:00+03:00     17255.50
2020-05-25 05:00:00+03:00     16389.07
2020-05-25 06:00:00+03:00     15595.52
2020-05-25 07:00:00+03:00     15333.30
2020-05-25 08:00:00+03:00     15489.26
2020-05-25 09:00:00+03:00     15814.02
2020-05-25 10:00:00+03:00     16507.11

En yuksek: 2025-07-28 14:00:00+03:00 -> 59504 MWh
                     date  consumption
2025-07-28 11:00:00+03:00     57125.13
2025-07-28 12:00:00+03:00     56630.00
2025-07-28 13:00:00+03:00     58044.49
2025-07-28 14:00:00+03:00     59503.67
2025-07-28 15:00:00+03:00     59366.13
2025-07-28 16:00:00+03:00     59130.85
2025-07-28 17:00:00+03:00     58128.60


## C. Birleştirme ve Marj Taraması

### C1. Tüketim + EAK + günlük sıcaklık

- Sıcaklık günlük olduğu için o günün tüm saatlerine aynı değer yayılıyor.
- NOAA verisi 22 Eylül 2026'da bittiği için, sonraki saatlerde sıcaklık eksik görünür; beklenen bir durum.
- İki marj tanımı yan yana hesaplanıyor:
  - Kapasite bazlı marj = (EAK − tüketim) / EAK
  - Yük bazlı marj = (EAK − tüketim) / tüketim (PJM IRM tanımı)

In [5]:
sic = pd.read_csv(os.path.join(KOK, "data", "ankara_tavg_gunluk.csv"), index_col=0, parse_dates=True)

b = tuketim[["date", "consumption"]].merge(
        eak_tum[["date", "toplam", "gunes", "ruzgar", "barajli", "dogalgaz"]], on="date", how="inner")
b["gun"] = b["date"].dt.tz_localize(None).dt.normalize()
b = b.merge(sic, left_on="gun", right_index=True, how="left")
b["marj_kapasite"] = (b["toplam"] - b["consumption"]) / b["toplam"]
b["marj_yuk"] = (b["toplam"] - b["consumption"]) / b["consumption"]
b["ay"] = b["date"].dt.month
b["saat"] = b["date"].dt.hour
print("Birlesik satir:", len(b), "| sicakligi eksik saat:", b["tavg_c"].isna().sum())

Birlesik satir: 91968 | sicakligi eksik saat: 120


### C2. Tam dönem marj taraması (ham veri)

In [6]:
print(b["marj_kapasite"].describe().round(3).to_string())
for esik in [0.30, 0.25, 0.20, 0.15]:
    print(f"Kapasite bazli marj < %{int(esik * 100)}: {(b['marj_kapasite'] < esik).sum()} saat")

print("\nEn dar 10 saat:")
print(b.nsmallest(10, "marj_kapasite")[["date", "consumption", "toplam", "gunes", "tavg_c", "marj_kapasite"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

alt = b[b["marj_kapasite"] < b["marj_kapasite"].quantile(0.01)]
print("\nEn dar %1 saatin aylara dagilimi:")
print(alt["ay"].value_counts().sort_index().to_string())
print("\nEn dar %1 saatin gun icindeki saatlere dagilimi:")
print(alt["saat"].value_counts().sort_index().to_string())

count    91968.000
mean         0.404
std          0.090
min          0.087
25%          0.340
50%          0.403
75%          0.472
max          0.712
Kapasite bazli marj < %30: 11820 saat
Kapasite bazli marj < %25: 3962 saat
Kapasite bazli marj < %20: 1041 saat
Kapasite bazli marj < %15: 131 saat

En dar 10 saat:
                     date  consumption    toplam    gunes  tavg_c  marj_kapasite
2024-07-23 15:00:00+03:00    57516.720 62987.270 1658.310  28.100          0.087
2024-07-23 14:00:00+03:00    57772.400 64076.070 1663.930  28.100          0.098
2016-12-19 11:00:00+03:00    41714.990 46270.160      NaN  -4.900          0.098
2024-07-22 14:00:00+03:00    57473.330 63809.410 1664.650  29.500          0.099
2021-08-04 14:00:00+03:00    55575.020 61958.540      NaN  29.200          0.103
2024-07-22 16:00:00+03:00    56579.780 63104.770 1650.870  29.500          0.103
2021-08-03 13:00:00+03:00    54110.370 60410.270      NaN  28.700          0.104
2016-12-19 10:00:00+03:00    40907.

## D. EAK Kırılma Analizi

### D1. Yıllık özet ve güneş sütununun tarihçesi

In [7]:
kaynaklar = ["dogalgaz", "ruzgar", "linyit", "tasKomur", "ithalKomur", "fuelOil", "jeotermal",
             "barajli", "nafta", "biokutle", "akarsu", "gunes", "diger"]

e = eak_tum.copy()
e["yil"] = e["date"].dt.year
ozet = e.groupby("yil").agg(
    toplam_ort=("toplam", "mean"), toplam_max=("toplam", "max"),
    gunes_dolu_oran=("gunes", lambda s: s.notna().mean()),
    gunes_max=("gunes", "max"), ruzgar_max=("ruzgar", "max"))
print(ozet.round(2).to_string())
print("\nGunes sutununun ilk dolu oldugu saat:", e.loc[e["gunes"].notna(), "date"].min())

ogle = e[e["date"].dt.hour == 12].set_index("date")["gunes"].resample("MS").mean()
print("\n12:00 gunes EAK, aylik ortalama (2023 ve sonrasi):")
print(ogle["2023":].round(0).to_string())

      toplam_ort  toplam_max  gunes_dolu_oran  gunes_max  ruzgar_max
yil                                                                 
2016    49843.49    56038.70             0.00        NaN     4417.72
2017    51421.33    57953.93             0.00        NaN     5138.61
2018    58219.57    63561.05             0.00        NaN     6629.30
2019    58322.62    64283.88             0.00        NaN     7424.91
2020    58100.62    63211.16             0.00        NaN     8644.77
2021    60194.75    66542.78             0.00        NaN    10366.72
2022    60804.69    71665.00             0.00        NaN    12115.10
2023    67384.45    74688.01             0.75    1779.75    12319.82
2024    64638.99    73761.52             1.00    1896.07    11642.75
2025    67085.58    85723.86             1.00   18022.79    13496.91
2026    69090.43    89038.16             1.00   20885.83    14742.86

Gunes sutununun ilk dolu oldugu saat: 2023-01-01 00:00:00+03:00

12:00 gunes EAK, aylik ortalama (2023

### D2. Kaynak bazında yıllık ortalamalar

In [8]:
yillik = e.groupby("yil")[kaynaklar + ["toplam"]].mean().round(0)
print(yillik.to_string())

      dogalgaz   ruzgar  linyit  tasKomur  ithalKomur  fuelOil  jeotermal  barajli  nafta  biokutle  akarsu   gunes   diger   toplam
yil                                                                                                                                 
2016   16150.0   3056.0  4512.0     913.0      5153.0    702.0      429.0  15158.0   12.0      83.0  3334.0     NaN   343.0  49843.0
2017   17047.0   3759.0  4396.0     668.0      5741.0    175.0      540.0  15132.0   12.0     120.0  3535.0     NaN   295.0  51421.0
2018   17901.0   5334.0  5281.0     129.0      7015.0    195.0      845.0  15383.0   12.0     276.0  5515.0     NaN   306.0  58220.0
2019   14451.0   6497.0  5163.0     309.0      6973.0    239.0     1075.0  16252.0   12.0     437.0  6513.0     NaN   379.0  58323.0
2020   13310.0   7730.0  4177.0     132.0      6953.0    219.0     1254.0  16391.0    9.0     635.0  6690.0     NaN   467.0  58101.0
2021   14846.0   9499.0  4515.0     129.0      6349.0    129.0     12

### D3. Sıçrama adayları: ne zaman, hangi saatte, hangi kaynaktan

Adaylar 23:00 ile 00:00 arasında toplanıyorsa, bu bir veri hatası değil, iki günün bildirimleri arasındaki geçiştir. Kaynak atfı, her sıçramada en büyük değişimi hangi kaynağın yaptığını gösterir.

In [9]:
fark = eak_tum["toplam"].diff()
esik = 4 * fark.abs().std()
aday = eak_tum[((fark > esik) & (fark.shift(-1) < -esik)) | ((fark < -esik) & (fark.shift(-1) > esik))]
print("Sicrama adayi sayisi:", len(aday))
print("Yillara gore:", aday["date"].dt.year.value_counts().sort_index().to_dict())
print("Saatlere gore:", aday["date"].dt.hour.value_counts().sort_index().to_dict())

fark_k = eak_tum[kaynaklar].diff()
suclu = fark_k.loc[aday.index].abs().idxmax(axis=1)
print("\nSicramaya en cok katkida bulunan kaynak:")
print(suclu.value_counts().to_string())

i = aday.index[0]
print("\nIlk ornegin komsu saatleri:")
print(eak_tum.loc[i - 3:i + 3, ["date", "toplam", "gunes", "ruzgar", "dogalgaz", "barajli"]].to_string(index=False))

Sicrama adayi sayisi: 46
Yillara gore: {2025: 44, 2026: 2}
Saatlere gore: {0: 5, 1: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 7, 8: 5, 10: 2, 11: 2, 12: 6, 13: 6, 14: 1, 15: 4, 16: 1, 19: 1, 23: 1}

Sicramaya en cok katkida bulunan kaynak:
dogalgaz    42
gunes        3
barajli      1

Ilk ornegin komsu saatleri:
                     date   toplam   gunes  ruzgar  dogalgaz  barajli
2025-04-19 20:00:00+03:00 60638.50 2224.96 7842.32  10533.23 18564.07
2025-04-19 21:00:00+03:00 60317.41 2211.30 7684.80  10627.68 18314.07
2025-04-19 22:00:00+03:00 59679.35 2209.48 7530.35  10532.55 17979.77
2025-04-19 23:00:00+03:00 63357.71 2208.92 7469.60  14506.26 17952.77
2025-04-20 00:00:00+03:00 57866.26 2103.21 7337.82   9480.85 17833.45
2025-04-20 01:00:00+03:00 62919.63 2099.80 7324.39  14557.78 17761.65
2025-04-20 02:00:00+03:00 63140.41 2099.83 7157.85  15045.05 17621.65


### D4. Darlık örüntüsü döneme göre değişiyor mu?

Hipotez: EAK güneşi kapsadıktan sonra öğle saatleri genişledi ve darlık akşama kaydı.

In [10]:
b["donem"] = (b["date"].dt.year >= 2025).map({False: "2016-2024", True: "2025-2026"})
for d, g in b.groupby("donem"):
    alt_d = g[g["marj_kapasite"] < g["marj_kapasite"].quantile(0.01)]
    print(f"\n{d}: saat={len(g)}, marj ort={g['marj_kapasite'].mean():.3f}, min={g['marj_kapasite'].min():.3f}")
    print("  En dar %1 saatin gun ici dagilimi:", alt_d["saat"].value_counts().sort_index().to_dict())


2016-2024: saat=76728, marj ort=0.405, min=0.087
  En dar %1 saatin gun ici dagilimi: {9: 11, 10: 50, 11: 109, 12: 55, 13: 86, 14: 149, 15: 128, 16: 100, 17: 40, 18: 11, 19: 5, 20: 11, 21: 8, 22: 4, 23: 1}

2025-2026: saat=15240, marj ort=0.399, min=0.146
  En dar %1 saatin gun ici dagilimi: {16: 2, 17: 12, 18: 33, 19: 30, 20: 44, 21: 23, 22: 7, 23: 2}


### D5. Bildirim yöntemi değişiklikleri

Üç soru:
1. Güneş 2023'te gece de ~1.570 MWh mi? Sabitse, bildirim o yıl kurulu güce dayanıyordu.
2. Rüzgârdaki 2024 düşüşü ve "diğer" sütunundaki düşüş hangi ayda başlıyor? Ani bir ay, yöntem değişikliğine işaret eder.
3. Mayıs 2026'daki güneş çukuru tam olarak ne zaman başlıyor?

In [11]:
# 1) Gunes gun ici profili: 2023 / 2024 / 2025
for gun in ["2023-06-15", "2024-06-15", "2025-06-15"]:
    g = e[e["date"].dt.strftime("%Y-%m-%d") == gun]
    p = g.set_index(g["date"].dt.hour)["gunes"]
    print(gun, "gunes 00/06/12/18/23:", p.reindex([0, 6, 12, 18, 23]).round(0).tolist())

# 2) Ruzgar ve diger: aylik ortalama, 2021-2024
aylik_vre = e.set_index("date")[["ruzgar", "diger"]].resample("MS").mean()
print("\nRuzgar ve diger, aylik ortalama (2021-2024):")
print(aylik_vre["2021":"2024"].round(0).to_string())

# 3) Mayis 2026 anomalisinin baslangici: 24 Nisan - 1 Mayis, 12:00
son_nisan = e[(e["date"] >= pd.Timestamp("2026-04-24T00:00:00+03:00")) &
              (e["date"] < pd.Timestamp("2026-05-02T00:00:00+03:00")) & (e["date"].dt.hour == 12)]
print("\nNisan sonu - Mayis basi, 12:00:")
print(son_nisan[["date", "gunes", "toplam"]].to_string(index=False))

2023-06-15 gunes 00/06/12/18/23: [1586.0, 1656.0, 1688.0, 1676.0, 1586.0]
2024-06-15 gunes 00/06/12/18/23: [19.0, 1594.0, 1664.0, 1594.0, 19.0]
2025-06-15 gunes 00/06/12/18/23: [2277.0, 5230.0, 16873.0, 6070.0, 2275.0]

Ruzgar ve diger, aylik ortalama (2021-2024):
                            ruzgar   diger
date                                      
2021-01-01 00:00:00+03:00   8713.0   775.0
2021-02-01 00:00:00+03:00   8798.0   879.0
2021-03-01 00:00:00+03:00   9049.0   920.0
2021-04-01 00:00:00+03:00   9235.0   987.0
2021-05-01 00:00:00+03:00   9393.0   999.0
2021-06-01 00:00:00+03:00   9519.0  1097.0
2021-07-01 00:00:00+03:00   9586.0   876.0
2021-08-01 00:00:00+03:00   9664.0   960.0
2021-09-01 00:00:00+03:00   9812.0   995.0
2021-10-01 00:00:00+03:00   9935.0  1033.0
2021-11-01 00:00:00+03:00  10039.0  1125.0
2021-12-01 00:00:00+03:00  10203.0  1167.0
2022-01-01 00:00:00+03:00  10314.0  1224.0
2022-02-01 00:00:00+03:00  10366.0  1229.0
2022-03-01 00:00:00+03:00  10458.0  1303.0
2022

### D6. Mayıs 2026 ve Ekim-Aralık 2023 kontrolleri

In [12]:
# Mayis 2026: ogle gunes EAK gun gun
may26 = e[(e["date"].dt.year == 2026) & (e["date"].dt.month == 5) & (e["date"].dt.hour == 12)]
print(may26[["date", "gunes", "toplam"]].to_string(index=False))

# Ekim-Aralik 2023: gunes bosken toplam kesintiye ugruyor mu?
e23 = e[e["date"].dt.year == 2023].set_index("date")
aylik23 = e23.resample("MS").agg({"toplam": "mean", "gunes": lambda s: s.notna().mean()})
print("\n2023 aylik toplam ve gunes dolu orani:")
print(aylik23.round(2).to_string())

                     date    gunes   toplam
2026-05-01 12:00:00+03:00  6905.07 63671.37
2026-05-02 12:00:00+03:00  5881.02 62079.20
2026-05-03 12:00:00+03:00  3698.32 61344.17
2026-05-04 12:00:00+03:00  5544.47 66908.34
2026-05-05 12:00:00+03:00  6145.92 66226.49
2026-05-06 12:00:00+03:00  7430.21 64127.82
2026-05-07 12:00:00+03:00  8393.09 67017.03
2026-05-08 12:00:00+03:00  8237.96 66002.27
2026-05-09 12:00:00+03:00  8134.33 65700.82
2026-05-10 12:00:00+03:00  8627.28 63778.88
2026-05-11 12:00:00+03:00  8211.40 62630.19
2026-05-12 12:00:00+03:00  8283.66 64791.43
2026-05-13 12:00:00+03:00  8449.65 64038.42
2026-05-14 12:00:00+03:00  7562.61 63649.09
2026-05-15 12:00:00+03:00  7592.01 62439.37
2026-05-16 12:00:00+03:00  8289.68 62210.41
2026-05-17 12:00:00+03:00  8304.83 64347.62
2026-05-18 12:00:00+03:00  7896.86 61773.46
2026-05-19 12:00:00+03:00  8190.24 62135.17
2026-05-20 12:00:00+03:00  6494.96 64309.06
2026-05-21 12:00:00+03:00  6452.15 63017.09
2026-05-22 12:00:00+03:00 16544.

## E. Temizlik, İşaretler ve Agent Eşikleri

### E1. Dönem ve sıçrama işaretleri

- `eak_donem`: Mart 2025 öncesi ve sonrası. EAK'nın kapsamı Mart 2025'te değişti; iki tanımın karışmaması için.
- `eak_sicrama`: D3'teki sıçrama adayı saatleri.

In [13]:
KIRILMA = pd.Timestamp("2025-03-01T00:00:00+03:00")
b["eak_donem"] = (b["date"] >= KIRILMA).map({False: "2016-04_2025-02", True: "2025-03_sonrasi"})
b["eak_sicrama"] = b["date"].isin(aday["date"]).astype(int)
print(b["eak_donem"].value_counts().to_string())
print("Sicrama isaretli saat:", int(b["eak_sicrama"].sum()))

eak_donem
2016-04_2025-02    78144
2025-03_sonrasi    13824
Sicrama isaretli saat: 46


### E2. Sıçrama düzeltmesi ve Mayıs 2026 anomali işareti

- Sıçrama saatlerinde toplam EAK boşaltılıp komşu saatlerden doğrusal olarak dolduruluyor (`toplam_duz`). Art arda gelen sıçrama saatleri de doğru ele alınıyor. Orijinal değer (`toplam`) korunuyor.
- Mayıs 2026 anomalisi silinmiyor, yalnızca işaretleniyor (`eak_anomali`). **`ANOM_BAS` değerini D5'teki Nisan sonu çıktısına göre gerekirse güncelleyin.**

In [14]:
e2 = eak_tum.copy()
e2["toplam_duz"] = e2["toplam"].where(~e2["date"].isin(aday["date"]))
e2["toplam_duz"] = e2["toplam_duz"].interpolate(limit_direction="both")

b = b.drop(columns=["toplam_duz"], errors="ignore").merge(e2[["date", "toplam_duz"]], on="date", how="left")

ANOM_BAS = pd.Timestamp("2026-05-01T00:00:00+03:00")   # D5 ciktisina gore guncelleyin
ANOM_BIT = pd.Timestamp("2026-05-22T00:00:00+03:00")   # 22 Mayis'ta gunes EAK'si Nisan seviyesine donuyor
b["eak_anomali"] = ((b["date"] >= ANOM_BAS) & (b["date"] < ANOM_BIT)).astype(int)

b["marj_kapasite_duz"] = (b["toplam_duz"] - b["consumption"]) / b["toplam_duz"]
b["marj_yuk_duz"] = (b["toplam_duz"] - b["consumption"]) / b["consumption"]
print("Duzeltilen sicrama saati:", int(b["eak_sicrama"].sum()), "| anomali saati:", int(b["eak_anomali"].sum()))
print("Duzeltme oncesi / sonrasi en dar marj (yeni donem):",
      round(b.loc[b["eak_donem"] == "2025-03_sonrasi", "marj_kapasite"].min(), 3), "/",
      round(b.loc[b["eak_donem"] == "2025-03_sonrasi", "marj_kapasite_duz"].min(), 3))

Duzeltilen sicrama saati: 46 | anomali saati: 504
Duzeltme oncesi / sonrasi en dar marj (yeni donem): 0.146 / 0.146


### E3. Temiz yeni dönemden Agent eşikleri

**Eşiklerin anlamı:** Türkiye için yayımlanmış bir MW eşiği bulunamadı; bulunan resmî eşikler frekans cinsinden. Bu yüzden eşikler, Mart 2025 sonrası temizlenmiş dönemin kapasite bazlı marj dağılımından türetiliyor. Bunlar **tarihsel olarak ne kadar nadir** bir durumda olunduğunu ölçer, fiziksel bir güvenilirlik sınırını değil.

**Faz 3 için önerilen karar eşleştirmesi:**

| Marj | Durum | Karar |
|---|---|---|
| ≥ %5'lik dilim | normal | `normal_operation` |
| %1 – %5'lik dilim arası | attention | `demand_response_alert` |
| < %1'lik dilim | attention, yüksek öncelik | `demand_response_alert` |
| ≤ 0 (tahmin kapasiteyi aşıyor) | action_required | `activate_reserve` |

In [15]:
temiz = b[(b["eak_donem"] == "2025-03_sonrasi") & (b["eak_anomali"] == 0)]
print("Temiz yeni donem saat:", len(temiz))

dilimler = [0.001, 0.01, 0.05, 0.10, 0.50]
karsilastirma = pd.DataFrame({
    "ham_kapasite": b.loc[b["eak_donem"] == "2025-03_sonrasi", "marj_kapasite"].quantile(dilimler),
    "temiz_kapasite": temiz["marj_kapasite_duz"].quantile(dilimler),
    "temiz_yuk": temiz["marj_yuk_duz"].quantile(dilimler),
})
print(karsilastirma.round(3).to_string())

q = temiz["marj_kapasite_duz"].quantile([0.01, 0.05])
esikler = {
    "yontem": "Mart 2025 sonrasi, sicramalari duzeltilmis ve Mayis 2026 anomalisi disarida birakilmis "
              "kapasite bazli marj dagilimi; resmi bir MW esigi olmadigi icin veri tabanli",
    "marj_tanimi": "(EAK - tuketim) / EAK",
    "cok_nadir_alti": round(float(q.loc[0.01]), 3),
    "nadir_alti": round(float(q.loc[0.05]), 3),
    "donem": [str(temiz["date"].min()), str(temiz["date"].max())],
    "saat": int(len(temiz)),
}
with open(os.path.join(KOK, "metrics", "agent_esikleri.json"), "w", encoding="utf-8") as f:
    json.dump(esikler, f, ensure_ascii=False, indent=2)
print(json.dumps(esikler, ensure_ascii=False, indent=2))

Temiz yeni donem saat: 13320
       ham_kapasite  temiz_kapasite  temiz_yuk
0.001         0.180           0.178      0.217
0.010         0.208           0.207      0.261
0.050         0.247           0.246      0.326
0.100         0.277           0.275      0.380
0.500         0.400           0.401      0.669
{
  "yontem": "Mart 2025 sonrasi, sicramalari duzeltilmis ve Mayis 2026 anomalisi disarida birakilmis kapasite bazli marj dagilimi; resmi bir MW esigi olmadigi icin veri tabanli",
  "marj_tanimi": "(EAK - tuketim) / EAK",
  "cok_nadir_alti": 0.207,
  "nadir_alti": 0.246,
  "donem": [
    "2025-03-01 00:00:00+03:00",
    "2026-09-27 23:00:00+03:00"
  ],
  "saat": 13320
}


### E4. Birleşik veri setini kaydetme

Kaydedilen sütunlar: tüketim, ham ve düzeltilmiş EAK, seçili kaynaklar, sıcaklık, ham ve düzeltilmiş marjlar, dönem, sıçrama ve anomali işaretleri.

In [16]:
kayit = ["date", "consumption", "toplam", "toplam_duz", "gunes", "ruzgar", "barajli", "dogalgaz",
         "tavg_c", "tavg_doldurma", "marj_kapasite", "marj_yuk", "marj_kapasite_duz", "marj_yuk_duz",
         "eak_donem", "eak_sicrama", "eak_anomali"]
b[kayit].to_csv(os.path.join(KOK, "data", "tr_birlesik_saatlik.csv"), index=False)
print("Kaydedildi:", len(b), "satir,", len(kayit), "sutun")

Kaydedildi: 91968 satir, 17 sutun
